In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.12 Electrons in a Periodic Potential: Bloch's Theorem and the Origin of Bands

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.12",
    title="Electrons in a Periodic Potential: Bloch's Theorem and the Origin of Bands",
    blurb="The free electron gas priced the heat of metals, the stiffness of potassium, and "
    "the mass of a dead star — and it cannot tell copper from diamond. The missing "
    "ingredient is the lattice. One symmetry argument shows that electrons in a "
    "crystal organize into bands separated by forbidden gaps; two independent "
    "computations build the same bands and meet; and a counting argument — two "
    "electrons per cell per band — finally divides the material world into metals "
    "and insulators. The statistics resumes in the next notebook, on the density of "
    "states built here.",
    difficulty="advanced",
    estimate="200–240 min",
)

## Notebook overview

Movement III's free electron gas is a triumph with a hole in it. Fresh from its report card —
the linear heat capacity and Pauli paramagnetism of [§7.10](fermi-gas-finite-temperature.ipynb), the compressibility of potassium and
the sound of sodium in [§7.9](fermi-gas-zero-temperature.ipynb), the mass of a dead star in [§7.11](white-dwarfs-chandrasekhar.ipynb) — it returns to Earth and must
confess: it predicts that *every* material containing electrons conducts. Nature disagrees, at
scale. Diamond carries four valence electrons per atom and insulates behind a 5.5 eV gap; the
room-temperature resistivities of real materials span roughly thirty orders of magnitude, the
widest dynamic range of any material property, and every decade of it is invisible to a theory
whose density of states never touches zero. The missing ingredient is not statistical. It is the
**lattice**.

So this notebook breaks the volume's rhythm, deliberately and on precedent: one excursion into
single-particle quantum mechanics — admitted the way Movement 0's mathematics was admitted, as
an arsenal the statistics cannot proceed without; after it, the statistics resumes ([§7.13](semiconductor-statistics.ipynb)).
The contract is narrow and the payoff is large. **Bloch's theorem** is derived as a Volume VI
symmetry move (a periodic potential commutes with the lattice translation; unitary operators
have unit-circle eigenvalues) and then *verified* on a discretized ring, complete with a taught
numerical trap: real eigenvectors from `eigh` hide the Bloch phases inside degenerate pairs, and
only diagonalizing the translation *within each degenerate subspace* recovers them, landing exactly on the allowed $k$-grid. **Kronig–Penney** makes bands exact from one transcendental
line, with the forbidden gaps located by root finding. The course-signature **rendezvous**
solves the *same* crystal by an unrelated second method, the plane-wave central equation, and the two meet; better, the *rate* at which they meet is the lesson: the singular comb converges
like $1/n_G$ while a smooth cosine potential converges at $n_G = 2$ to ten digits, and that
measured contrast is precisely why the world's plane-wave density-functional codes run on
pseudopotentials (the outward horizon: real 3D bands, DFT, and the MMM course). **Nearly-free
electrons** explain what a gap *is* — Bragg reflection in stationary form, the zone-boundary
degeneracy split by exactly $2|V_1|$. And **tight binding** approaches the same bands from the
atomic limit, with a meta-recognition the course has earned: the finite-difference Laplacian
used since Volume 0 *is* a tight-binding hopping matrix, and every continuum calculation so far
has secretly been a crystal.

The destination is arithmetic. Bloch counting gives $2N$ states per band — **two electrons per
cell per band**, and a filled band is verifiably *inert* (its group velocities sum to zero and
Pauli forbids the rearrangement a field requests). Odd electrons per cell leave a band
half-filled: a Fermi surface, a metal. Even electrons per cell can fill bands exactly and strand
the Fermi level in a gap: an insulator. Thirty orders of magnitude of resistivity reduce to a
parity. The gapped density of states, with its van Hove edges and its exact zeros, is the
object this notebook hands to [§7.13](semiconductor-statistics.ipynb), where the chemical potential moves into the gap and
Fermi–Dirac, unchanged since [§7.7](bose-einstein-fermi-dirac.ipynb), becomes the physics of semiconductors.

> **Conventions (this notebook).** Working units $\hbar = m = a = 1$ (energies in
> $\hbar^2/ma^2$). Reduced-zone scheme with $k \in [-\pi/a, \pi/a]$; the allowed
> grid on an $N_c$-cell ring is $k = 2\pi m/N_ca$; the spin factor 2 rides every count. Basis
> sizes $n_G$ are stated per computation, with the convergence rule measured (not assumed) in
> the rendezvous. Finite-difference rings wrap *both* corners; `brentq` brackets come from a
> fine pre-scan (the missed-bracket trap is stated); group-velocity sums exclude the duplicated
> periodic endpoint.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: $\|[H, T_a]\|$ at rounding level and the Bloch phases on the exact $2\pi m/N_c$ grid;
> Kronig–Penney's band edges against the transcendental condition; the central equation meeting
> the transcendental answer at the measured $\sim 1/n_G$ rate while the cosine converges at
> $n_G = 2$; the zone-boundary gap equal to $2|V_1|$; the tight-binding ring matching
> $-2t\cos ka$ at $10^{-15}$; the van Hove histogram against the closed form; and a filled
> band's velocity sum at zero. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*.
>
> **Scope (the framing constraint, binding).** This is a notebook *within a volume on quantum
> statistical mechanics*: one quantum input, the periodic potential, built to the depth the
> statistics requires, and no further. Horizons point *outward* only: real 3D band structures
> and Fermi surfaces, density-functional theory and pseudopotentials (the MMM course is the
> reader's next home for electronic structure at scale), graphene's linear bands in one breath.
> No Wannier functions, no topology, no transport theory here. See Ashcroft & Mermin
> (Chs. 8–10); Kittel (Ch. 7); Kronig & Penney (1931). Cross-reference [§6.6](../06-quantum-mechanics/pauli-uncertainty.ipynb)/[§6.7](../06-quantum-mechanics/time-evolution.ipynb) (commuting
> operators and unitaries), [§6.21](../06-quantum-mechanics/perturbation-fine-structure.ipynb) (degenerate perturbation theory, applied at the zone boundary),
> [§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb) (the box states, now periodized), [§7.3](statistical-toolkit.ipynb) (the DOS machinery), [§7.9](fermi-gas-zero-temperature.ipynb)–[§7.11](white-dwarfs-chandrasekhar.ipynb) (the confession's
> triumphs), and forward to [§7.13](semiconductor-statistics.ipynb) (Fermi–Dirac in the gap) and [§7.16](phonons-debye.ipynb) (the same Bloch counting for
> phonons).

## Theory in brief

### The confession

Three lines of triumph: the free gas priced the electronic heat capacity and spin
susceptibility of real metals to tens of percent with nothing fitted ([§7.10](fermi-gas-finite-temperature.ipynb)), the
compressibility of the alkalis and the speed of sound in sodium ([§7.9](fermi-gas-zero-temperature.ipynb)), and the limiting mass of
white dwarfs to four percent against Sirius B ([§7.11](white-dwarfs-chandrasekhar.ipynb)). One line of failure: it cannot produce an insulator (a gas with $g(\varepsilon) > 0$ everywhere always has states to rearrange, so it always conducts) while diamond insulates behind a 5.5 eV gap and the resistivity of matter
spans $\sim 10^{30}$ from the best metals to the best insulators. This notebook supplies the one
missing input, $V(x+a) = V(x)$, on the Movement-0 precedent: an arsenal notebook, because the
statistics cannot proceed without it. The statistics resumes in [§7.13](semiconductor-statistics.ipynb).

### Bloch's theorem, by symmetry

A periodic potential commutes with the lattice translation $T_a$: $[H, T_a] = 0$, so energy
eigenstates can be chosen as $T_a$ eigenstates (the commuting-operator move of [§6.6](../06-quantum-mechanics/pauli-uncertainty.ipynb)/[§6.7](../06-quantum-mechanics/time-evolution.ipynb)). $T_a$ is
unitary, its eigenvalues lie on the unit circle, and writing them $e^{ika}$:

```{math}
:label: eq-band-bloch
\psi_k(x+a) = e^{ika}\,\psi_k(x)
\qquad\Longleftrightarrow\qquad
\psi_k(x) = e^{ikx}\,u_k(x),\quad u_k(x+a) = u_k(x),
```

Bloch waves, labelled by a **crystal momentum** $k$ defined only modulo $G = 2\pi/a$ — fold
everything into the **Brillouin zone** $[-\pi/a, \pi/a]$. (Crystal momentum is not momentum:
the lattice can absorb recoil in units of $\hbar G$.) The counting that the whole notebook
marches toward: a periodic ring of $N$ cells allows $k = 2\pi m/Na$, exactly $N$ values per band; with spin, **$2N$ states per band: two electrons per cell per band**.

### Bloch verified, with a taught trap

On a discretized ring ($N_c$ cells, $M$ points each) the translation is a permutation matrix,
and the verification is direct:

```{math}
:label: eq-bloch-verified
\|[H, T_a]\| \sim 10^{-13},
\qquad
\text{eigenphases of } T_a\big|_{\text{each degenerate subspace}} = \frac{2\pi m}{N_c}.
```

The trap is worth its lesson: `numpy.linalg.eigh` returns *real* eigenvectors, which inside each
degenerate $\pm k$ pair are cosine/sine mixtures, whose naive expectation $\langle v|T|v\rangle$
is real, and the extracted "phases" collapse to $0$ and $\pi$. The fix is general: restrict $T$
to each degenerate subspace (`sub.T @ T @ sub`) and diagonalize the small unitary: degenerate subspaces demand *simultaneous* diagonalization, a lesson the course states once and uses
forever.

### Kronig–Penney: bands, exactly

For the Dirac-comb crystal $V = (\hbar^2P/ma)\sum_n\delta(x - na)$, matching $\psi$ and its
kinked derivative across one cell and imposing Bloch's condition gives one transcendental line:

```{math}
:label: eq-kronig-penney
\cos(ka) = \cos(qa) + P\,\frac{\sin(qa)}{qa},
\qquad
E = \frac{\hbar^2q^2}{2m}.
```

The left side lives in $[-1, 1]$; energies where the right side escapes that corridor are
**forbidden** — gaps, from one line of algebra. Band edges are the roots of $|F(q)| = 1$
(located by `brentq` on a pre-scanned grid; for $P = 3$ the first band runs $[1.953, 4.935]$
with the next at $[9.458, 19.74]$, in $\hbar^2/ma^2$). The limits bracket the subject: $P \to 0$
recovers the free continuum; $P \to \infty$ pinches the bands onto isolated-well levels: the atomic limit, tight binding's home.

### The rendezvous, and the pseudopotential moral

Expand $\psi$ in plane waves $e^{i(k+G)x}$ and the Schrödinger equation becomes the **central
equation** — a matrix eigenproblem per $k$:

```{math}
:label: eq-central-equation
H_{GG'} = \delta_{GG'}\,\frac{(k+G)^2}{2} + V_{G-G'},
\qquad
\text{comb: } V_G = P\ \text{for every }G .
```

A delta function has *every* Fourier component equal, so the comb's matrix is a free diagonal
plus a constant, and its plane-wave convergence is slow: the lowest band's error falls like
$\sim 1/n_G$ (measured: $3.4\times10^{-2} \to 1.75\times10^{-3}$ for $n_G = 10 \to 200$). A
smooth cosine potential, by contrast, converges at $n_G = 2$ to ten digits. Two unrelated
methods meet on one band structure — and the *rate* of the meeting is the outward moral: sharp
potentials demand enormous plane-wave baskets, which is exactly why plane-wave
density-functional codes replace the singular ionic potential with smooth **pseudopotentials**
(real 3D bands, DFT, and the MMM course: the reader's next home for this, at scale).

### Nearly-free electrons: the gap is Bragg

Keep a single Fourier component, $V_{\pm G_1} = V_1$ (the cosine crystal). At the zone boundary
$k = \pi/a$ the free waves $e^{\pm i\pi x/a}$ are degenerate, and the degenerate machinery of [§6.21](../06-quantum-mechanics/perturbation-fine-structure.ipynb) applies verbatim: the $2\times2$ secular problem mixes them into standing waves:

```{math}
:label: eq-nfe-gap
\psi_\pm \propto \cos,\ \sin\!\big(\pi x/a\big),
\qquad
E_\pm = \frac{\pi^2}{2a^2} \pm |V_1|
\quad\Longrightarrow\quad
\Delta E = 2|V_1| .
```

The cosine piles density *on* the ion rows and the sine piles it *off* — the crystal cannot
propagate the wave it Bragg-reflects, and the gap is what standing still costs. Verified
numerically: ratios $1.0000, 1.0000, 0.9998$ at $V_1 = 0.05, 0.2, 0.5$, the small drift being
honest second order.

### Tight binding, and the course's confession-in-reverse

From the atomic limit: $N$ orbitals on a ring, hopping $t$ to neighbours (Ashcroft &
Mermin, Ch. 10, develops the tight-binding method in full),

```{math}
:label: eq-tight-binding
H = -t\,(S + S^{\mathsf T})
\quad\Longrightarrow\quad
E(k) = -2t\cos(ka),
\qquad
\text{bandwidth } 4t,
\quad
m^* = \frac{\hbar^2}{2ta^2}\ \text{at the bottom}.
```

And the meta-note, stated with pleasure: the finite-difference Laplacian this course has used
since Volume 0 *is* this hopping matrix with $t = \hbar^2/2m\Delta x^2$. Every "continuum"
calculation so far has secretly been a tight-binding crystal whose bandwidth $4t$ was kept far
above the physics; when it isn't, discretization artifacts *are* band-structure artifacts.

### The band density of states: van Hove

The machinery of [§7.3](statistical-toolkit.ipynb) pointed at a band. From $E = -2t\cos k$, inverting $|dk/dE|$:

```{math}
:label: eq-van-hove
g(E) = \frac{1}{\pi\sqrt{4t^2 - E^2}},
```

divergent at the band edges — the 1D **van Hove singularities** (in 2D a logarithm, in 3D
square-root onsets: named horizons), and *exactly zero* in the gaps. A density of states with
gaps is precisely the object [§7.13](semiconductor-statistics.ipynb) feeds into Fermi–Dirac; nothing in Movements 0–II had one.

### Metal or insulator: the counting

The payoff, by arithmetic, assembled from Bloch counting and the evenness of $E(k)$
(Kittel, Ch. 7, closes on the same counting):

```{math}
:label: eq-metal-insulator
\sum_{k\in\text{BZ}} v_k = \frac{1}{\hbar}\sum_k \frac{\partial E}{\partial k} = 0
\quad(\text{a filled band is inert}),
\qquad
\begin{cases}
\text{odd }e^-/\text{cell} \Rightarrow \text{half-filled band} \Rightarrow \text{METAL}\\
\text{even }e^-/\text{cell} \Rightarrow \text{filled bands, } \varepsilon_F \text{ in a gap} \Rightarrow \text{INSULATOR}.
\end{cases}
```

A filled band carries no current: its velocities cancel identically ($E(k)$ is even) and Pauli
forbids the rearrangement an applied field requests. Sodium's one electron per cell half-fills a
band — a Fermi surface, a metal. Diamond's even count fills bands exactly — the Fermi level
strands in the gap. Honest caveats, named: divalent magnesium is metallic because bands
*overlap* in $\ge$2D (the 1D chain cannot show it), and the band-top curvature is *negative* —
a nearly-filled band responds as positive carriers of mass $|m^*|$: **holes**, the concept [§7.13](semiconductor-statistics.ipynb)
runs on.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: working units ħ = m = a = 1 (energies in ħ^2/ma^2);
# reduced zone k ∈ [−π, π]; allowed grid k = 2πm/N_c on an N_c-cell ring; spin
# factor 2 in every count. FD rings wrap BOTH corners; brentq brackets come from a
# fine pre-scan; velocity sums exclude the duplicated periodic endpoint; basis sizes
# n_G are stated per computation.


def ring_hamiltonian(Nc, M, Vfunc):
    """The periodic-ring Hamiltonian by finite differences: N_c cells, M points per cell.

    H = −(1/2)∂_x^2 + V(x) on N_p = N_c·M points with spacing Δx = 1/M, the Laplacian
    wrapped at BOTH corners (H[0, −1] and H[−1, 0] — forgetting one silently opens the
    ring and destroys the translation symmetry this notebook is about). Per Exercise
    6's meta-note, this matrix IS a tight-binding crystal with t = 1/(2Δx^2); its
    bandwidth is kept far above every energy we trust.

    Parameters
    ----------
    Nc : int
        Number of unit cells on the ring.
    M : int
        Grid points per cell (Δx = 1/M).
    Vfunc : callable
        The periodic potential V(x), vectorized.

    Returns
    -------
    numpy.ndarray
        The (N_c·M) × (N_c·M) real symmetric Hamiltonian.
    """
    Np = Nc * M
    dx = 1.0 / M
    x = np.arange(Np) * dx
    t_fd = 1.0 / (2.0 * dx**2)
    H = np.diag(2.0 * t_fd + Vfunc(x))
    off = -t_fd * np.ones(Np - 1)
    H += np.diag(off, 1) + np.diag(off, -1)
    H[0, -1] = H[-1, 0] = -t_fd  # both wrap corners: the ring stays a ring
    return H


def translation(Np, M):
    """The one-cell translation operator T_a on the N_p-point ring, as a permutation.

    T = numpy.roll(numpy.eye(N_p), M, axis=1): (Tψ)(x_i) = ψ(x_i + a), a pure
    permutation and hence exactly unitary. Its commutator with any periodic-ring H is
    zero up to rounding — the numerical statement of V(x + a) = V(x).

    Parameters
    ----------
    Np : int
        Total grid points.
    M : int
        Points per cell (the shift).

    Returns
    -------
    numpy.ndarray
        The N_p × N_p permutation matrix.
    """
    return np.roll(np.eye(Np), M, axis=1)


def bloch_phases(H, T, degeneracy_tol=1e-8):
    """Extract the Bloch eigenphases of T_a, respecting degenerate subspaces.

    numpy.linalg.eigh returns REAL eigenvectors; inside each degenerate ±k pair these
    are cos/sin mixtures whose naive ⟨v|T|v⟩ is real (phases 0/π only — the trap
    Exercise 2 demonstrates). The honest extraction groups levels by |ΔE| <
    degeneracy_tol, restricts T to each group (sub.T @ T @ sub), and diagonalizes the
    small unitary with numpy.linalg.eigvals — simultaneous diagonalization, done
    subspace by subspace.

    Parameters
    ----------
    H : numpy.ndarray
        The ring Hamiltonian (real symmetric).
    T : numpy.ndarray
        The translation operator.
    degeneracy_tol : float, optional
        Energy tolerance for grouping degenerate levels (default 1e-8).

    Returns
    -------
    tuple
        (energies, phases): eigenenergies and the matching T_a eigenphases in [0, 2π).
    """
    E, V = np.linalg.eigh(H)
    phases = np.empty_like(E)
    i = 0
    while i < E.size:
        j = i + 1
        while j < E.size and abs(E[j] - E[i]) < degeneracy_tol:
            j += 1
        sub = V[:, i:j]
        T_sub = sub.T @ T @ sub  # the restriction of T to the degenerate subspace
        lam = np.linalg.eigvals(T_sub)
        phases[i:j] = np.sort(np.mod(np.angle(lam), 2.0 * np.pi))
        i = j
    return E, phases


def kp_condition(q, P):
    """The Kronig–Penney function F(q) = cos(q) + P sin(q)/q (a = 1 units).

    Bloch's condition demands cos(ka) = F(q): energies E = q^2/2 are ALLOWED where
    |F| ≤ 1 (some real k exists) and FORBIDDEN where |F| > 1 — bands and gaps from
    one transcendental line (eq-kronig-penney). The q → 0 limit is 1 + P (finite).

    Parameters
    ----------
    q : float or numpy.ndarray
        The internal wavenumber (E = q^2/2).
    P : float
        The comb strength.

    Returns
    -------
    float or numpy.ndarray
        F(q).
    """
    q = np.asarray(q, dtype=float)
    return np.where(np.abs(q) < 1e-10, 1.0 + P, np.cos(q) + P * np.sinc(q / np.pi))


def kp_band_edges(P, q_max=15.0, n_scan=30001):
    """Band edges of the Kronig–Penney crystal: brentq on |F(q)| = 1 after a fine pre-scan.

    |F| − 1 oscillates, and a coarse bracket hunt silently drops edges — so the scan
    grid is fine (n_scan points to q_max) and every sign change of |F| − 1 is handed
    to brentq. Returned as a list of (E_bottom, E_top) per band, E = q^2/2. The q = 0
    endpoint (F = 1 + P > 1) starts the scan inside a forbidden region, so edges pair
    up as (enter corridor, leave corridor).

    Parameters
    ----------
    P : float
        The comb strength.
    q_max : float, optional
        Scan limit in q (default 15.0 — three bands for P = 3).
    n_scan : int, optional
        Pre-scan resolution (default 30001).

    Returns
    -------
    list of tuple
        Band intervals [(E_lo, E_hi), ...] in units ħ^2/ma^2.
    """
    q = np.linspace(1e-6, q_max, n_scan)
    g = np.abs(kp_condition(q, P)) - 1.0
    roots = []
    for i in range(q.size - 1):
        if g[i] * g[i + 1] < 0:
            roots.append(
                brentq(
                    lambda qq: abs(kp_condition(qq, P)) - 1.0,
                    q[i],
                    q[i + 1],
                    xtol=1e-12,
                )
            )
    edges = [r**2 / 2.0 for r in roots]
    return [(edges[i], edges[i + 1]) for i in range(0, len(edges) - 1, 2)]


def central_equation(k, V_of_G, nG):
    """The plane-wave (central-equation) bands at crystal momentum k: eigh of H_{GG'}.

    H_{GG'} = δ_{GG'}(k+G)^2/2 + V_{G−G'} over the reciprocal lattice G = 2πm,
    m = −n_G..n_G — a (2n_G+1)-dimensional eigenproblem per k (eq-central-equation).
    The basis size n_G is the honesty dial: Exercise 4 MEASURES the convergence (the
    comb needs ~1/n_G patience, the cosine converges at n_G = 2), which is the
    pseudopotential moral in one table.

    Parameters
    ----------
    k : float
        Crystal momentum in the reduced zone.
    V_of_G : callable
        V_G as a function of the integer index m (G = 2πm).
    nG : int
        Half-width of the plane-wave basis.

    Returns
    -------
    numpy.ndarray
        Sorted band energies at this k.
    """
    m_idx = np.arange(-nG, nG + 1)
    G = 2.0 * np.pi * m_idx
    H = np.diag((k + G) ** 2 / 2.0).astype(float)
    for i, mi in enumerate(m_idx):
        for j, mj in enumerate(m_idx):
            H[i, j] += V_of_G(
                mi - mj
            )  # V_0 rides the diagonal: a constant energy offset
    return np.linalg.eigvalsh(H)


def tight_binding_ring(N, t):
    """The N-site tight-binding ring H = −t(S + S^T), built from numpy.roll shifts.

    The hopping matrix whose spectrum is −2t cos(2πm/N) (eq-tight-binding) — and,
    per the course's confession-in-reverse, exactly the finite-difference Laplacian
    of every notebook since Volume 0, with t = ħ^2/2mΔx^2 and the site energy
    2t absorbed into the zero.

    Parameters
    ----------
    N : int
        Number of sites (cells) on the ring.
    t : float
        The hopping amplitude.

    Returns
    -------
    numpy.ndarray
        The N × N hopping Hamiltonian.
    """
    S = np.roll(np.eye(N), 1, axis=1)
    return -t * (S + S.T)

## Exercise 1 — The confession, quantified

The free gas's report card: five triumphs and one failure wide enough to hold thirty orders of
magnitude. (The framing exercise; short by design.)

1. Recap in a table what the free gas got right ($\gamma$, $\chi_{\text{Pauli}}$, $B$ of the
   alkalis, Bohm–Staver, $M_{\text{Ch}}$; one line each with the notebook of origin).
2. State the failure with data: diamond's gap (5.5 eV) and the $\sim10^{30}$ resistivity span
   from best metal to best insulator (representative values stated).
3. Argue in prose why *no* free-gas mechanism can produce an insulator, locating the missing
   ingredient in $V(x)$.
4. State the notebook's contract (the framing constraint, in the course's voice): one arsenal
   notebook of single-particle quantum mechanics, then the statistics resumes.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    span > 1e25 and gap_diamond_eV / 0.02585 > 100,
    "the confession, quantified: ~30 decades of resistivity and a 213 k_BT gap, invisible to the free gas",
    f"span 10^{np.log10(span):.0f}, gap {gap_diamond_eV / 0.02585:.0f} k_BT",
)

## Exercise 2 — Bloch's theorem, by symmetry — then verified with a trap

Translation symmetry organizes the spectrum; the computer confirms it, once we respect
degeneracy. Cite {eq}`eq-band-bloch`, {eq}`eq-bloch-verified`.

1. Derive Bloch's theorem from $[H, T_a] = 0$ and the unitarity of $T_a$ (the [§6.6](../06-quantum-mechanics/pauli-uncertainty.ipynb)/[§6.7](../06-quantum-mechanics/time-evolution.ipynb) move),
   defining crystal momentum and the Brillouin zone.
2. Build the periodic-ring $H$ (`ring_hamiltonian`, both wrap corners) and
   $T_a$ = `numpy.roll(numpy.eye(Np), M, axis=1)`; verify $\|[H, T_a]\| \lesssim 10^{-12}$.
3. Demonstrate the trap: naive phases $\angle\langle v|T|v\rangle$ on `eigh`'s real eigenvectors
   return only $0/\pi$; then extract correctly with `bloch_phases` (the degenerate-subspace
   restriction) and confirm the phases land on the grid $2\pi m/N_c$.
4. Count (prose + one line): $N$ cells $\Rightarrow$ $N$ $k$-values per band $\Rightarrow$ $2N$
   states with spin — two electrons per cell per band, the number the whole notebook is
   marching toward.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    comm_norm < 1e-11,
    "the symmetry, numerically: the periodic-ring H commutes with T_a at rounding level",
    f"‖[H, T_a]‖ = {comm_norm:.1e}",
)
validate.close(
    band1_phases,
    grid,
    "Bloch verified: the subspace-extracted eigenphases land exactly on the allowed k-grid",
    atol=1e-6,
)
validate.check(
    distinct_naive.size <= 2,
    "and the trap is real: naive phases on real eigh vectors collapse to {0, π}",
    f"distinct naive values: {distinct_naive}",
)

## Exercise 3 — Kronig–Penney: bands from one transcendental line

An exactly solvable crystal, with its forbidden gaps located by root-finding. Cite
{eq}`eq-kronig-penney`.

1. Derive the condition $\cos(ka) = \cos(qa) + P\sin(qa)/qa$ for the Dirac comb by matching
   across one cell.
2. Use `kp_band_edges(P)` (fine pre-scan of $|F(q)| - 1$, then `brentq` per bracket; the missed-bracket trap stated); verify for $P = 3$ the first bands $[1.953, 4.935]$ and
   $[9.458, 19.74]$.
3. Plot $F(q)$ with the $\pm1$ corridor (allowed bands where the curve stays inside, gaps where it escapes) and the resulting $E(k)$ in the reduced zone.
4. Take the limits: $P \to 0$ (free continuum recovered) and $P$ large (bands pinching onto
   isolated-well levels — the atomic limit; show the first band's width shrink).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    list(bands_KP[0]) + list(bands_KP[1]),
    [1.9532, 4.9348, 9.4577, 19.7392],
    "Kronig–Penney at P = 3: the first two bands, exactly where the transcendental line puts them",
    rtol=1e-3,
)
validate.check(
    width_P30 < 0.3 * width_P3,
    "the atomic limit: the first band pinches toward the isolated-well level as P grows",
    f"width {width_P3:.3f} (P=3) → {width_P30:.3f} (P=30)",
)

## Exercise 4 — The rendezvous — and why DFT uses pseudopotentials

The same crystal by a second, independent method; the convergence rate is the lesson. Cite
{eq}`eq-central-equation`.

1. Derive the central equation and assemble $H_{GG'}$ for the comb ($V_G = P$ for all $G$ —
   state why every Fourier component is equal).
2. Diagonalize with `eigh` at $k = 0.3, 1.0, 2.94$ and compare the lowest band with the
   transcendental dispersion (agreement at the $10^{-2}$–$10^{-3}$ level at $n_G = 25$, honestly
   reported).
3. Measure the convergence: the lowest-band error for $n_G = 10 \to 200$ falling $\sim 1/n_G$;
   contrast the cosine potential, converged at $n_G = 2$ to ten digits.
4. Read the moral outward (prose): plane waves hate sharp potentials, the measured reason
   plane-wave density-functional codes replace ionic singularities with smooth pseudopotentials;
   the MMM course named as where the reader meets this at scale.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    comb_errors[200] < comb_errors[10] / 10.0 and comb_errors[200] < 5e-3,
    "the comb's plane-wave error falls ~1/n_G: sharp potentials demand enormous baskets",
    f"{comb_errors[10]:.1e} → {comb_errors[200]:.1e} for n_G 10 → 200",
)
validate.check(
    cos_spread < 1e-10,
    "the smooth cosine converges at n_G = 2 to ten digits — the pseudopotential moral, measured",
    f"spread {cos_spread:.1e}",
)

## Exercise 5 — The gap is Bragg

Nearly-free electrons: one Fourier component, one degeneracy, one gap of exactly $2|V_1|$. Cite
{eq}`eq-nfe-gap`.

1. Assemble the central equation for the cosine crystal (only $V_{\pm1} = V_1$) with the Setup
   `central_equation` and diagonalize at the zone boundary $k = \pi/a$ (`numpy.linalg.eigvalsh`).
2. Verify gap $= 2|V_1|$ (ratios $1.0000, 1.0000, 0.9998$ at $V_1 = 0.05, 0.2, 0.5$; the small drift attributed to second order).
3. Recover the same $2|V_1|$ by the degenerate perturbation theory of [§6.21](../06-quantum-mechanics/perturbation-fine-structure.ipynb) on the two zone-boundary
   waves (the $2\times2$ secular problem), and exhibit the standing-wave eigenstates with
   density on/off the ion rows.
4. Tell it as Bragg (prose): the crystal reflects the wave it cannot propagate, and the gap is
   the price of standing still; plot the reduced-zone bands with the gaps marked.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    gap_ratios,
    [1.0000, 1.0000, 0.9998],
    "the zone-boundary gap is 2|V_1| — Bragg in stationary form, drifting only at second order",
    rtol=1e-3,
)
validate.check(
    abs(abs(r_lower) - 1.0) < 1e-3
    and abs(abs(r_upper) - 1.0) < 1e-3
    and r_lower * r_upper < 0,
    "the eigenstates are the ± standing waves: density on the ion rows, and off them",
    f"component ratios {r_lower:+.3f}, {r_upper:+.3f}",
)

## Exercise 6 — Tight binding — and what the course has been doing all along

The atomic limit's band, and a meta-recognition about every finite-difference calculation since
Volume 0. Cite {eq}`eq-tight-binding`.

1. Build the $N$-site ring hopping matrix with `tight_binding_ring` and diagonalize
   (`numpy.linalg.eigh`); verify the spectrum equals $-2t\cos(2\pi m/N)$ to $\sim10^{-15}$,
   bandwidth $4t$.
2. Extract $m^* = \hbar^2/2ta^2$ from the band-bottom curvature (Taylor and a numerical second
   difference agreeing).
3. Make the identification (prose + one formula): the course's finite-difference Laplacian is
   *exactly* this matrix with $t = \hbar^2/2m\Delta x^2$: every continuum calculation has been
   a tight-binding crystal whose bandwidth was kept far above the physics; state the failure
   mode.
4. Connect the two limits (prose): Kronig–Penney at large $P$ pinched toward these flat atomic
   bands; nearly-free at small $V_1$ barely dented the parabola — one band structure, approached
   from both ends.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    spec_eigh,
    spec_analytic,
    "tight binding: the ring's eigh spectrum equals −2t cos(2πm/N) at machine precision",
    atol=1e-12,
)
validate.close(
    m_star,
    1.0 / (2.0 * t_hop),
    "the band-bottom effective mass m* = ħ²/2ta², by Taylor and by second difference",
    rtol=1e-6,
)

## Exercise 7 — The band density of states: van Hove

The machinery of [§7.3](statistical-toolkit.ipynb) pointed at a band, with divergent edges and exact zeros. Cite
{eq}`eq-van-hove`.

1. Derive $g(E) = (1/\pi)/\sqrt{4t^2 - E^2}$ from $|dk/dE|$ for the tight-binding band.
2. Histogram the eigenvalues of a large ring (`numpy.histogram(..., density=True)`; the
   spectrum from the `eigh`-certified closed form at $N = 20000$) and verify against the closed
   form away from the edges (bin-average comparison via the exact $\int g\,dE = \arcsin(E/2t)/\pi$;
   the integrable edge divergence handled as stated).
3. Mark the two van Hove edge singularities and name their higher-dimensional shapes (2D
   logarithm, 3D square-root onsets) as the horizon.
4. State the object built (prose): a density of states with *gaps* — the precise input [§7.13](semiconductor-statistics.ipynb)
   feeds to Fermi–Dirac.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    dos_err < 0.01,
    "the band DOS matches the bin-averaged (1/π)/√(4t²−E²) on the interior — edges excluded as stated",
    f"worst interior deviation {dos_err:.2%}",
)

## Exercise 8 — Metal or insulator, by counting

The confession answered with arithmetic: a filled band is inert, and the parity of electrons
per cell decides. Cite {eq}`eq-metal-insulator`.

1. Verify the inert-band theorem numerically: the analytic $v_k = 2t\sin(ka)$ summed with
   `numpy.sum` over the $2\pi m/N$ grid vanishes (the periodic-endpoint trap stated), and argue
   the Pauli half.
2. For the cosine crystal's computed bands, place the Fermi level for 1, 2, and 3 electrons per
   cell and classify by Bloch counting on the computed band extrema — `central_equation` bands
   on the $k$-grid, extrema by `numpy.ndarray.min`/`max` per band; an odd count half-fills a
   band, an even count is insulating exactly when the next band's bottom lies above the filled
   band's top: metal (mid-band), insulator (in-gap), metal.
3. State the honest caveats: divalent magnesium is metallic via band *overlap* (a $\ge$2D effect
   the chain cannot exhibit); and compute the band-top curvature: $m^* < 0$, introducing the
   hole as the carrier [§7.13](semiconductor-statistics.ipynb) will count.
4. Close the arc (prose): thirty orders of magnitude of resistivity reduce to whether the
   highest band is half-full or exactly full.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    v_sum,
    0.0,
    "a filled band carries no current: the zone-summed group velocity vanishes",
    atol=1e-12,
)
validate.check(
    verdicts[1] == "metal" and verdicts[2] == "insulator" and verdicts[3] == "metal",
    "the parity rule: 1, 2, 3 electrons per cell classify metal / insulator / metal",
)
validate.close(
    gap_2e,
    2.0 * V1_plot,
    "and the insulating gap read off the computed extrema is the Bragg gap of Exercise 5",
    rtol=2e-3,
)
validate.check(
    d2E_top < 0.0,
    "the band-top curvature is negative: the hole, handed to §7.13",
    f"d²E/dk² = {d2E_top:.3f}",
)

## Exercise 9 — The one quantum input

This notebook broke the volume's rhythm on purpose — one excursion into single-particle quantum
mechanics, granted the way Movement 0's mathematics was granted, because the statistics could
not proceed without it. What it bought: a theorem that organizes every electron in every crystal
by symmetry alone; bands and gaps computed twice by unrelated methods that met in the middle
(and taught, in their meeting, why the world's electronic-structure codes smooth their
potentials); a gap explained as Bragg reflection standing still; the discovery that the course's
own numerical Laplacians were tight-binding crystals all along; and a counting rule — two per
cell per band — that splits the material world in two. The free gas's confession is answered:
diamond insulates because its electrons exactly fill what the lattice offers, and a full band,
like a full Fermi sea, cannot respond.

There is a quiet symmetry between this notebook's two limits. Start from free electrons and the
lattice opens gaps in a continuum; start from bound atoms and hopping smears levels into bands.
The truth is indifferent to where one starts, which is, perhaps, the deepest thing a pair of
converging approximations can teach.

Now the volume does what it came to do: the next notebook sets the chemical potential loose in
the gap we built, and Fermi–Dirac, unchanged since [§7.7](bose-einstein-fermi-dirac.ipynb), turns band structure into the physics
of semiconductors.

## Notebook summary

The volume's one quantum input, built to the depth the statistics requires and handed back.

- **The confession** (framing): the free gas's five triumphs ([§7.9](fermi-gas-zero-temperature.ipynb)–[§7.11](white-dwarfs-chandrasekhar.ipynb)) against its one failure (no mechanism for an insulator), quantified by diamond's 5.5 eV gap and the $\sim10^{30}$
  resistivity span.
- **Bloch by symmetry** {eq}`eq-band-bloch`, verified {eq}`eq-bloch-verified`: $[H, T_a] = 0$ at
  $10^{-13}$ on the ring; the eigenphases extracted by *degenerate-subspace* diagonalization
  (the naive-real-eigenvector trap demonstrated first) land exactly on $2\pi m/N_c$; the count (two electrons per cell per band) flagged as the destination.
- **Kronig–Penney** {eq}`eq-kronig-penney`: bands $[1.953, 4.935]$ and $[9.458, 19.74]$ at
  $P = 3$ from `brentq` on the pre-scanned corridor condition; the free and atomic limits taken.
- **The rendezvous** {eq}`eq-central-equation`: the central equation meets the transcendental
  answer, with the *rate* as the lesson — the comb's $\sim1/n_G$ against the cosine's ten
  digits at $n_G = 2$: the pseudopotential moral, measured (the outward bridge to DFT and the
  MMM course).
- **The gap is Bragg** {eq}`eq-nfe-gap`: $2|V_1|$ at the zone boundary (ratios $1.0000$,
  $1.0000$, $0.9998$), recovered by the two-by-two of [§6.21](../06-quantum-mechanics/perturbation-fine-structure.ipynb), with the standing waves exhibited.
- **Tight binding** {eq}`eq-tight-binding`: $-2t\cos ka$ at $10^{-15}$, $m^* = \hbar^2/2ta^2$. And the meta-note: the course's finite-difference Laplacian has been a crystal all along.
- **Van Hove** {eq}`eq-van-hove`: the band DOS with divergent edges and *exact zeros* — the
  gapped $g(E)$ that is the input of [§7.13](semiconductor-statistics.ipynb).
- **The parity rule** {eq}`eq-metal-insulator`: a filled band is inert ($\sum v_k = 0$ at
  $10^{-14}$, Pauli locking the rest); fillings 1/2/3 per cell classify metal/insulator/metal;
  band overlap and the negative-mass hole stated as the honest caveats and the handoff.

The statistics resumes next door, on the density of states built here.

## Outlook

- **Fermi–Dirac in the gap ([§7.13](semiconductor-statistics.ipynb)).** Intrinsic and doped semiconductors, holes, the law of mass action: the statistics, resumed on this notebook's object.
- **Outward horizons, named.** Real 3D band structures and Fermi surfaces; density-functional
  theory and pseudopotentials — the MMM course as the reader's next home for electronic
  structure at scale; graphene's linear bands in one breath.
- **The same counting, other waves ([§7.16](phonons-debye.ipynb)).** Bloch's theorem for lattice vibrations: phonon
  bands, and the Debye physics the Einstein solid missed.
- **Cross-reference** [§6.6](../06-quantum-mechanics/pauli-uncertainty.ipynb)/[§6.7](../06-quantum-mechanics/time-evolution.ipynb) (the symmetry move), [§6.21](../06-quantum-mechanics/perturbation-fine-structure.ipynb) (the zone-boundary two-by-two), [§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb)
  (the box, periodized), [§7.3](statistical-toolkit.ipynb) (the DOS machinery), [§7.9](fermi-gas-zero-temperature.ipynb)–[§7.11](white-dwarfs-chandrasekhar.ipynb) (the confession's triumphs), [§5.1](../05-classical-stat-mech/counting.ipynb)
  (counting, again decisive).

In [ ]:
from ecp.style import footer

footer()